# K-FAC NGD on Continual Permuted MNIST
#
**K-FAC Natural Gradient Descent** + implicit BatchNorm mechanisms on Continual Permuted MNIST.
#
**Benchmark**: Online Permuted MNIST (CBP paper setup).
Network: FC 784 → 2000 × 5 → 10, ReLU.
800 tasks × 60 000 examples/task (full MNIST train), batch size 32.

## 1. Imports & Setup

In [ ]:
import os, sys, time, pickle, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Optimizer
from torch.utils.data import DataLoader, TensorDataset
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'serif'
matplotlib.rcParams['font.size'] = 11

_LOP_ROOT = "/kaggle/input/datasets/nguyenlamphuquy/lop-src/lop"
if _LOP_ROOT not in sys.path:
    sys.path.insert(0, _LOP_ROOT)

from lop.nets.deep_ffnn import DeepFFNN
from lop.algos.bp import Backprop
from lop.algos.cbp import ContinualBackprop
from lop.utils.miscellaneous import nll_accuracy as accuracy
from lop.utils.miscellaneous import compute_matrix_rank_summaries

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Metrics

In [ ]:
NUM_HIDDEN_LAYERS = 5
NUM_FEATURES      = 2000
INPUT_SIZE        = 784
CLASSES_PER_TASK  = 10

@torch.no_grad()
def compute_avg_weight_magnitude(net):
    n, s = 0, 0.0
    for p in net.parameters():
        n += p.numel()
        s += torch.sum(torch.abs(p)).item()
    return s / n if n > 0 else 0.0

@torch.no_grad()
def compute_dead_neurons(net, probe_x):
    """Count dead neurons (zero activation across all probe samples) per hidden layer."""
    net.eval()
    _, hidden_acts = net.predict(probe_x)
    dead_per_layer = []
    for act in hidden_acts:
        dead_per_layer.append((act.abs().sum(dim=0) == 0).sum().item())
    return dead_per_layer

def compute_stable_rank(sv):
    if len(sv) == 0: return 0
    sorted_sv = np.flip(np.sort(sv))
    cumsum = np.cumsum(sorted_sv) / np.sum(sv)
    return int(np.sum(cumsum < 0.99) + 1)

def compute_effective_rank(singular_values):
    if len(singular_values) == 0: return 0.0
    norm_sv = singular_values / np.sum(np.abs(singular_values))
    entropy = 0.0
    for p in norm_sv:
        if p > 0.0:
            entropy -= p * np.log(p)
    return float(np.e ** entropy)

@torch.no_grad()
def compute_dormant_neurons_ffnn(net, probe_x, threshold=0.01):
    """Compute dormant neuron fraction per hidden layer for DeepFFNN.
    Returns (aggregate_frac, per_layer_fracs, last_hidden_act_numpy).
    """
    net.eval()
    _, hidden_acts = net.predict(probe_x)
    per_layer_frac = []
    total_d, total_n = 0, 0
    last_act = None
    for i, act in enumerate(hidden_acts):
        alive_score = (act != 0).float().mean(dim=0)  # shape: [H]
        n_units = act.shape[1]
        dormant = (alive_score < threshold).sum().item()
        per_layer_frac.append(dormant / n_units if n_units > 0 else 0.0)
        total_d += dormant; total_n += n_units
        if i == len(hidden_acts) - 1:
            last_act = act.cpu().numpy()
    agg_frac = total_d / total_n if total_n > 0 else 0.0
    return agg_frac, per_layer_frac, last_act

def compute_stable_rank_from_activations(act):
    """Stable rank = number of SVs needed to capture 99% of sum(|σ|).
    Same formula as CBP's compute_abs_approximate_rank / compute_stable_rank."""
    from scipy.linalg import svd
    if act is None: return 0
    if act.ndim > 2: act = act.reshape(act.shape[0], -1)
    if act.shape[0] == 0 or act.shape[1] == 0: return 0
    try:
        sv = svd(act, compute_uv=False, lapack_driver="gesvd")
        return compute_stable_rank(sv)
    except: return 0

def compute_effective_rank_from_activations(act):
    """Effective rank via Shannon entropy of normalised singular values.
    Same formula as ngd_leakyReLU.py."""
    from scipy.linalg import svd
    if act is None: return 0.0
    if act.ndim > 2: act = act.reshape(act.shape[0], -1)
    if act.shape[0] == 0 or act.shape[1] == 0: return 0.0
    try:
        sv = svd(act, compute_uv=False, lapack_driver="gesvd")
        return compute_effective_rank(sv)
    except: return 0.0

print("✓ Metrics defined")

## 3. MNIST Data Loading (mirrors CBP repo)

In [ ]:
MNIST_CACHE = "data/mnist_"

def _build_mnist_cache(cache_path: str = MNIST_CACHE):
    os.makedirs(os.path.dirname(cache_path) if os.path.dirname(cache_path) else ".", exist_ok=True)
    tfm = transforms.Compose([transforms.ToTensor()])
    train_ds = torchvision.datasets.MNIST(root="data", train=True,  transform=tfm, download=True)
    test_ds  = torchvision.datasets.MNIST(root="data", train=False, transform=tfm, download=True)

    def _load_all(ds):
        loader = DataLoader(ds, batch_size=len(ds), shuffle=True)
        imgs, labels = next(iter(loader))
        return imgs.flatten(start_dim=1), labels   # (N, 784), (N,)

    print("  Loading train …", end=" ")
    x, y = _load_all(train_ds)
    print("done.  Loading test …", end=" ")
    x_test, y_test = _load_all(test_ds)
    print("done.")

    with open(cache_path, 'wb+') as f:
        pickle.dump([x, y, x_test, y_test], f)
    print(f"  Cached at '{cache_path}'")
    return x, y, x_test, y_test


def get_mnist(cache_path: str = MNIST_CACHE):
    if os.path.isfile(cache_path):
        with open(cache_path, 'rb+') as f:
            x, y, x_test, y_test = pickle.load(f)
    else:
        print(f"Cache '{cache_path}' not found — building …")
        x, y, x_test, y_test = _build_mnist_cache(cache_path)
    return x, y, x_test, y_test


os.makedirs("data", exist_ok=True)
print("Loading MNIST …")
x_mnist, y_mnist, x_test_mnist, y_test_mnist = get_mnist()
print(f"  Train : {x_mnist.shape}  labels: {y_mnist.shape}")
print(f"  Test  : {x_test_mnist.shape}  labels: {y_test_mnist.shape}")

IMAGES_PER_CLASS  = 6000
EXAMPLES_PER_TASK = IMAGES_PER_CLASS * CLASSES_PER_TASK   # 60 000
print(f"\n  examples_per_task (change_after) = {EXAMPLES_PER_TASK}")
print("✓ MNIST ready  (CBP paper format)")

## 4. K-FAC Natural Gradient Descent Optimizer
#
Kronecker-Factored Approximate Curvature (K-FAC) — efficient NGD.
Approximates Fisher as F ≈ A ⊗ G per layer.

In [ ]:
from torch.optim import Optimizer as _Optimizer

class KFACOptimizer(_Optimizer):
    """
    K-FAC Natural Gradient Descent — clean implementation.

    Approximates Fisher Information Matrix per layer as F ≈ A ⊗ G:
        A = EMA of E[a a^T]   (input/activation covariance)
        G = EMA of E[g g^T]   (output gradient covariance)
    Natural gradient:  δW = G_inv · ∇W · A_inv

    Implicit BatchNorm mechanisms (all in optimizer, no arch change):
    ─────────────────────────────────────────────────────────────────
    1. AVS  (Activation Variance Scaling): normalise x by running std
       BEFORE computing A.  A_eff = E[x̂ x̂^T] where x̂ = x / std(x).
       → A_eff ≈ correlation matrix (diag≈1), condition number drops.
       → Replicates the VARIANCE part of BN (BNP — Sheng & Zhang 2022,
         JMLR 2022).  Without AVS, one large-variance input dimension
         dominates A → all natural gradient steps are biased toward it.

    2. OPC  (Online Pre-activation Centering): after each nat-grad step
       nudge each hidden bias: b[i] -= opc_lr × E[z_i].
       → Keeps E[Wx+b] ≈ 0 per neuron.
       → Replicates the MEAN part of BN without batch statistics.
       → Direct counter to ReLU neuron death (pre-act mean drifting < 0).

    3. CWN  (Centered Weight Normalization): after each weight update,
       zero-center each output row: W[i] -= mean(W[i]).
       → Kills the radial gradient → makes Theorem 4.33 hold for plain MLP.
       → Replicates BN's zero-mean-weight property (Huang et al. ICCV 2017).
       → In NF-ResNets this is how the mean-subtraction of BN is replaced.

    4. NaP  (Normalize-and-Project): after CWN, clip each row to a fixed
       target norm: W[i] *= nap_norm / ||W[i]||.
       → Prevents weight norm growth → stable effective learning rate.
       → Replicates BN's scale-invariance property (Lyle et al. NeurIPS 2024).
       → Makes the Fisher-dominated regime reachable (Theorem 4.33 floor).

    CWN + NaP together = Weight Standardization (Qiao et al. 2019) applied
    as an optimizer post-step rather than as a network reparametrization.
    """

    def __init__(self, model, lr=0.01, damping=1e-3, weight_decay=0,
                 T_inv=100, alpha=0.95, max_grad_norm=1.0,
                 adapt_damping=True,
                 damping_adaptation_interval=5,
                 damping_adaptation_decay=0.99,
                 min_damping=1e-4,
                 damping_decrease_rho=0.85,
                 damping_increase_rho=0.35,
                 # ── Implicit BN mechanisms ──────────────────────────
                 avs=False,        # Activation Variance Scaling
                 opc_lr=0.0,       # Online Pre-activation Centering lr
                 cwn=False,        # Centered Weight Normalization
                 nap_norm=0.0,     # Normalize-and-Project target row-norm
                 ):
        self.model        = model
        self.damping      = damping
        self._init_damp   = damping
        self.weight_decay = weight_decay
        self.T_inv        = T_inv
        self.alpha        = alpha
        self.max_grad_norm = max_grad_norm
        self.steps        = 0

        # Adaptive damping (Levenberg-Marquardt rho)
        self.adapt_damping  = adapt_damping
        self._damp_interval = damping_adaptation_interval
        self._omega         = damping_adaptation_decay ** damping_adaptation_interval
        self._min_damping   = min_damping
        self._rho_dec       = damping_decrease_rho
        self._rho_inc       = damping_increase_rho
        self._prev_loss     = float('nan')
        self._qmodel_change = float('nan')
        self._rho           = float('nan')

        # Implicit BN
        self.avs      = avs
        self.opc_lr   = opc_lr
        self.cwn      = cwn
        self.nap_norm = nap_norm

        # Internal state
        self._modules = {}   # name -> nn.Module
        self._stats   = {}   # name -> {'A': Tensor, 'G': Tensor}
        self._inv     = {}   # name -> {'A_inv': Tensor, 'G_inv': Tensor}
        self._preact  = {}   # name -> EMA E[z]    (OPC)
        self._act_std = {}   # name -> EMA std(x)  (AVS)
        self._hooks   = []

        defaults = dict(lr=lr)
        super().__init__(model.parameters(), defaults)

        self._register_hooks()

        mech = []
        if avs:        mech.append("AVS (act-var scaling)")
        if opc_lr > 0: mech.append(f"OPC (lr={opc_lr})")
        if cwn:        mech.append("CWN (weight centering)")
        if nap_norm>0: mech.append(f"NaP (row-norm={nap_norm})")
        mech_str = ", ".join(mech) if mech else "none"
        print(f"  K-FAC tracking {len(self._modules)} layers | implicit-BN: {mech_str}")
        if adapt_damping:
            print(f"  Adaptive damping ON (interval={damping_adaptation_interval}, "
                  f"omega={self._omega:.4f}, rho=[{damping_increase_rho},{damping_decrease_rho}])")

    # ──────────────────────────────────────────────────────────────
    # Hooks
    # ──────────────────────────────────────────────────────────────

    def _register_hooks(self):
        for name, mod in self.model.named_modules():
            if isinstance(mod, (nn.Linear, nn.Conv2d)):
                self._modules[name] = mod
                self._stats[name]   = {'A': None, 'G': None}
                h1 = mod.register_forward_hook(self._fwd_hook(name, mod))
                h2 = mod.register_full_backward_hook(self._bwd_hook(name, mod))
                self._hooks += [h1, h2]

    def _fwd_hook(self, name, mod):
        def hook(m, inp, out):
            if not m.training:
                return
            with torch.no_grad():
                x = inp[0].detach()
                if isinstance(m, nn.Conv2d):
                    x = F.unfold(x, m.kernel_size, dilation=m.dilation,
                                 padding=m.padding, stride=m.stride)
                    x = x.permute(0, 2, 1).reshape(-1, x.size(1))
                elif x.dim() > 2:
                    x = x.reshape(-1, x.size(-1))

                # AVS: normalise x by running std before computing A
                if self.avs:
                    bs = x.std(dim=0).clamp(min=1e-6)
                    prev = self._act_std.get(name)
                    if prev is None:
                        self._act_std[name] = bs.clone()
                    else:
                        prev.mul_(self.alpha).add_(bs, alpha=1-self.alpha)
                    x = x / self._act_std[name].unsqueeze(0)

                if m.bias is not None:
                    ones = torch.ones(x.size(0), 1, device=x.device)
                    x = torch.cat([x, ones], dim=1)

                cov_a = x.t().mm(x) / x.size(0)
                s = self._stats[name]
                if s['A'] is None:
                    s['A'] = cov_a
                else:
                    s['A'].mul_(self.alpha).add_(cov_a, alpha=1-self.alpha)

                # OPC: track EMA E[z] for bias centering
                if self.opc_lr > 0 and isinstance(m, nn.Linear):
                    z = out.detach()
                    if z.dim() > 2:
                        z = z.reshape(-1, z.size(-1))
                    bm = z.mean(0)
                    prev = self._preact.get(name)
                    if prev is None:
                        self._preact[name] = bm.clone()
                    else:
                        prev.mul_(self.alpha).add_(bm, alpha=1-self.alpha)
        return hook

    def _bwd_hook(self, name, mod):
        def hook(m, grad_input, grad_output):
            if not m.training:
                return
            with torch.no_grad():
                g = grad_output[0].detach()
                if isinstance(m, nn.Conv2d):
                    g = g.permute(0, 2, 3, 1).reshape(-1, g.size(1))
                elif g.dim() > 2:
                    g = g.reshape(-1, g.size(-1))
                cov_g = g.t().mm(g) / g.size(0)
                s = self._stats[name]
                if s['G'] is None:
                    s['G'] = cov_g
                else:
                    s['G'].mul_(self.alpha).add_(cov_g, alpha=1-self.alpha)
        return hook

    # ──────────────────────────────────────────────────────────────
    # Curvature inversion
    # ──────────────────────────────────────────────────────────────

    def _invert_factors(self):
        sqrt_d = self.damping ** 0.5
        for name, s in self._stats.items():
            A, G = s['A'], s['G']
            if A is None or G is None:
                continue
            try:
                A_d = A + sqrt_d * torch.eye(A.size(0), device=A.device)
                G_d = G + sqrt_d * torch.eye(G.size(0), device=G.device)
                self._inv[name] = {
                    'A_inv': torch.linalg.inv(A_d),
                    'G_inv': torch.linalg.inv(G_d),
                }
            except RuntimeError:
                pass

    # ──────────────────────────────────────────────────────────────
    # Natural gradient
    # ──────────────────────────────────────────────────────────────

    def _compute_nat_grads(self):
        updates  = {}
        g_dot_ng = 0.0
        for name, mod in self._modules.items():
            if mod.weight.grad is None:
                continue
            gw    = mod.weight.grad
            has_b = mod.bias is not None and mod.bias.grad is not None

            if name in self._inv:
                Ai, Gi = self._inv[name]['A_inv'], self._inv[name]['G_inv']
                if isinstance(mod, nn.Conv2d):
                    g2d = gw.reshape(gw.size(0), -1)
                    if has_b:
                        g2d = torch.cat([g2d, mod.bias.grad.unsqueeze(1)], 1)
                    ng  = Gi.mm(g2d).mm(Ai)
                    ngw = ng[:, :-1].reshape_as(gw) if has_b else ng.reshape_as(gw)
                    ngb = ng[:, -1] if has_b else None
                else:
                    g2d = torch.cat([gw, mod.bias.grad.unsqueeze(1)], 1) if has_b else gw
                    ng  = Gi.mm(g2d).mm(Ai)
                    ngw = ng[:, :-1] if has_b else ng
                    ngb = ng[:, -1]  if has_b else None
            else:
                ngw = gw.clone()
                ngb = mod.bias.grad.clone() if has_b else None

            g_dot_ng += (gw * ngw).sum().item()
            if has_b and ngb is not None:
                g_dot_ng += (mod.bias.grad * ngb).sum().item()
            updates[name] = (ngw, ngb)
        return updates, g_dot_ng

    # ──────────────────────────────────────────────────────────────
    # Adaptive damping
    # ──────────────────────────────────────────────────────────────

    def _adapt_damping(self, loss_value):
        if not self.adapt_damping:
            return
        if (self.steps + 1) % self._damp_interval == 0:
            self._prev_loss = loss_value
        if (self.steps % self._damp_interval == 0) and self.steps > 0:
            if math.isnan(self._prev_loss) or math.isnan(self._qmodel_change):
                return
            rho = (loss_value - self._prev_loss) / (self._qmodel_change - 1e-12)
            self._rho = rho
            old = self.damping
            if   rho > self._rho_dec: self.damping = max(self.damping * self._omega, self._min_damping)
            elif rho < self._rho_inc: self.damping = self.damping / self._omega
            if self.damping != old:
                self._invert_factors()

    # ──────────────────────────────────────────────────────────────
    # Implicit BN post-step
    # ──────────────────────────────────────────────────────────────

    @torch.no_grad()
    def _apply_opc(self):
        """b[i] -= opc_lr * E[z_i]  for hidden layers only."""
        names    = list(self._modules.keys())
        out_name = names[-1] if names else None
        for name, mod in self._modules.items():
            if name == out_name or not isinstance(mod, nn.Linear) or mod.bias is None:
                continue
            mean = self._preact.get(name)
            if mean is not None:
                mod.bias.data.add_(mean, alpha=-self.opc_lr)

    @torch.no_grad()
    def _apply_cwn(self):
        names = list(self._modules.keys())
        output_layer_name = names[-1]  # Exclude output layer
        for name, mod in self._modules.items():
            if name == output_layer_name:
                continue  # ← THÊM DÒNG NÀY
            if isinstance(mod, nn.Linear):
                W = mod.weight.data
                W.sub_(W.mean(dim=1, keepdim=True))

    @torch.no_grad()
    def _apply_nap(self):
        """W[i] *= nap_norm / ||W[i]||  — clip rows to target norm."""
        for mod in self._modules.values():
            if isinstance(mod, nn.Linear):
                W     = mod.weight.data
                norms = W.norm(dim=1, keepdim=True).clamp(min=1e-6)
                W.mul_(self.nap_norm / norms)

    # ──────────────────────────────────────────────────────────────
    # Resets
    # ──────────────────────────────────────────────────────────────

    def reset_stats(self):
        """Soft reset at task boundary. Keeps G/A warm-start."""
        self.steps       = 0
        self.damping     = self._init_damp
        self._prev_loss  = float('nan')
        self._qmodel_change = float('nan')
        self._preact.clear()

    def hard_reset_stats(self):
        """Full reset including G/A. Use only after weight reinitialization."""
        for name in self._stats:
            self._stats[name] = {'A': None, 'G': None}
        self._inv.clear()
        self._preact.clear()
        self._act_std.clear()
        self.steps       = 0
        self.damping     = self._init_damp
        self._prev_loss  = float('nan')
        self._qmodel_change = float('nan')

    # ──────────────────────────────────────────────────────────────
    # Main step
    # ──────────────────────────────────────────────────────────────

    @torch.no_grad()
    def step(self, closure=None, loss_value=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        if loss_value is not None:
            self._adapt_damping(loss_value)

        if self.steps % self.T_inv == 0:
            self._invert_factors()

        lr = self.param_groups[0]['lr']
        updates, g_dot_ng = self._compute_nat_grads()

        # Gradient clipping
        total_sq = sum(
            ngw.norm().item()**2 + (ngb.norm().item()**2 if ngb is not None else 0.0)
            for ngw, ngb in updates.values()
        )
        clip = 1.0
        if self.max_grad_norm > 0 and total_sq**0.5 > self.max_grad_norm:
            clip = self.max_grad_norm / (total_sq**0.5 + 1e-6)

        # Quadratic model prediction (for damping adaptation)
        if self.adapt_damping and ((self.steps + 1) % self._damp_interval == 0):
            self._qmodel_change = -0.5 * lr * clip * g_dot_ng

        # Weight decay
        if self.weight_decay > 0:
            f = 1.0 - lr * self.weight_decay
            for mod in self._modules.values():
                mod.weight.data.mul_(f)
                if mod.bias is not None:
                    mod.bias.data.mul_(f)

        # Natural gradient updates
        for name, (ngw, ngb) in updates.items():
            mod = self._modules[name]
            mod.weight.data.add_(ngw, alpha=-lr * clip)
            if ngb is not None and mod.bias is not None:
                mod.bias.data.add_(ngb, alpha=-lr * clip)

        # SGD fallback for untracked params
        tracked = {id(m.weight) for m in self._modules.values()}
        tracked |= {id(m.bias) for m in self._modules.values() if m.bias is not None}
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None or id(p) in tracked:
                    continue
                if self.weight_decay > 0:
                    p.data.mul_(1.0 - lr * self.weight_decay)
                p.data.add_(p.grad, alpha=-lr)

        # ── Implicit BN post-step (order matters: OPC → CWN → NaP) ──
        if self.opc_lr  > 0:  self._apply_opc()
        if self.cwn and (self.steps % 10 == 0):           self._apply_cwn()
        if self.nap_norm > 0:  self._apply_nap()

        self.steps += 1
        return loss

print("✓ KFACOptimizer defined (clean K-FAC + 4 implicit-BN mechanisms: AVS, OPC, CWN, NaP)")

## 7. Configs

In [ ]:
SEED            = 42
SAVE_EVERY_N_TASKS = 80
TIME_LIMIT_SECONDS = 11.5 * 3600

# CBP paper constants (permuted_mnist/online_expr.py)
NUM_TASKS       = 800
CHANGE_AFTER    = 60_000
MINI_BATCH_SIZE = 32
STEPS_PER_TASK  = CHANGE_AFTER // MINI_BATCH_SIZE   # 1875
PROBE_SIZE      = 2000

_NGD_BASE = dict(
    lr=0.025, damping=1e-3, weight_decay=0, T_inv=50, alpha=0.95,
    # Implicit BatchNorm mechanisms — all disabled by default (pure K-FAC baseline)
    avs=False,      # Activation Variance Scaling
    opc_lr=0.025,     # Online Pre-activation Centering
    cwn=True,      # Centered Weight Normalization
    nap_norm=0.0,   # Normalize-and-Project
)

_CBP_BASE = dict(
    step_size=0.003,
    opt='sgd',
    replacement_rate=1e-4,      # Nature 2024 paper value
    decay_rate=0.99,
    maturity_threshold=100,
    util_type='adaptable_contribution',
    accumulate=True,
)

CONFIGS = {
    'ngd_kfac': {
        **_NGD_BASE,
    },
    'cbp': {
        **_CBP_BASE,
    },
}

METHODS_TO_RUN = ['ngd_kfac', 'cbp']

RESULTS_DIR = os.path.join('permuted_mnist_results', 'comparison')
CKPT_DIR    = os.path.join(RESULTS_DIR, 'checkpoints')
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"✓ Config: {NUM_TASKS} tasks × {STEPS_PER_TASK} steps/task, "
      f"batch={MINI_BATCH_SIZE}, network: {INPUT_SIZE}→{NUM_FEATURES}×{NUM_HIDDEN_LAYERS}→{CLASSES_PER_TASK}")

## 8. Build Optimizer

In [ ]:
def build_optimizer(config, model):
    return KFACOptimizer(
        model,
        lr           = config['lr'],
        damping      = config.get('damping', 1e-3),
        weight_decay = config.get('weight_decay', 0),
        T_inv        = config.get('T_inv', 100),
        alpha        = config.get('alpha', 0.95),
        # Implicit BN mechanisms
        avs          = config.get('avs',     False),
        opc_lr       = config.get('opc_lr',  0.0),
        cwn          = config.get('cwn',     False),
        nap_norm     = config.get('nap_norm', 0.0),
    )

print("✓ build_optimizer defined")

## 9. Training Loop — K-FAC NGD

In [ ]:
def _ckpt_path(method_name: str) -> str:
    return os.path.join(CKPT_DIR, f"ckpt_{method_name}.pt")


def run_ngd(method_name, config):
    """K-FAC Natural Gradient Descent on Permuted MNIST."""
    print(f"\n{'='*70}")
    print(f"  {method_name} — Permuted MNIST ({NUM_TASKS} tasks)")
    print(f"{'='*70}")

    wall_clock_start = time.time()
    torch.manual_seed(SEED); np.random.seed(SEED)

    # Build network: DeepFFNN 784 → 2000 × 5 → 10
    net = DeepFFNN(
        input_size        = INPUT_SIZE,
        num_features      = NUM_FEATURES,
        num_outputs       = CLASSES_PER_TASK,
        num_hidden_layers = NUM_HIDDEN_LAYERS,
    ).to(device)

    optimizer = build_optimizer(config, net)
    loss_fn = F.cross_entropy

    # Metric tensors
    total_iters       = NUM_TASKS * STEPS_PER_TASK
    accuracies        = torch.zeros(total_iters, dtype=torch.float)
    train_accuracies  = torch.zeros(NUM_TASKS, dtype=torch.float)
    test_accuracies   = torch.zeros(NUM_TASKS, dtype=torch.float)
    weight_mag_log    = torch.zeros((total_iters, NUM_HIDDEN_LAYERS + 1), dtype=torch.float)
    ranks             = torch.zeros((NUM_TASKS, NUM_HIDDEN_LAYERS), dtype=torch.float)
    effective_ranks   = torch.zeros((NUM_TASKS, NUM_HIDDEN_LAYERS), dtype=torch.float)
    approximate_ranks = torch.zeros((NUM_TASKS, NUM_HIDDEN_LAYERS), dtype=torch.float)
    dead_neurons      = torch.zeros((NUM_TASKS, NUM_HIDDEN_LAYERS), dtype=torch.float)
    all_stable_rank   = torch.zeros(NUM_TASKS, dtype=torch.float)
    all_effective_rank = torch.zeros(NUM_TASKS, dtype=torch.float)
    all_dormant_frac  = torch.zeros(NUM_TASKS, dtype=torch.float)
    task_metrics      = {'task_acc': [], 'task_train_acc': [], 'task_test_acc': [], 'task_id': [], 'avg_weight_mag': []}

    # Checkpoint resume
    ckpt_file  = _ckpt_path(method_name)
    start_task = 0
    global_iter = 0
    if os.path.isfile(ckpt_file):
        print(f"  Loading checkpoint: {ckpt_file}")
        ckpt = torch.load(ckpt_file, map_location=device, weights_only=False)
        net.load_state_dict(ckpt['model']); optimizer.load_state_dict(ckpt['optimizer'])
        accuracies        = ckpt.get('accuracies',         accuracies)
        train_accuracies  = ckpt.get('train_accuracies',   train_accuracies)
        test_accuracies   = ckpt.get('test_accuracies',    test_accuracies)
        weight_mag_log    = ckpt.get('weight_mag_log',     weight_mag_log)
        ranks             = ckpt.get('ranks',              ranks)
        effective_ranks   = ckpt.get('effective_ranks',    effective_ranks)
        approximate_ranks = ckpt.get('approximate_ranks',  approximate_ranks)
        dead_neurons      = ckpt.get('dead_neurons',       dead_neurons)
        all_stable_rank   = ckpt.get('all_stable_rank',    all_stable_rank)
        all_effective_rank = ckpt.get('all_effective_rank', all_effective_rank)
        all_dormant_frac  = ckpt.get('all_dormant_frac',   all_dormant_frac)
        task_metrics      = ckpt.get('task_metrics',       task_metrics)
        start_task        = ckpt['task'] + 1
        global_iter       = start_task * STEPS_PER_TASK
        print(f"  ✓ Resumed from task {start_task}")
        del ckpt; torch.cuda.empty_cache()
    else:
        print("  (no checkpoint — training from scratch)")

    def save_checkpoint(task_idx, reason="periodic"):
        ckpt_data = {
            'task': task_idx, 'model': net.state_dict(),
            'optimizer': optimizer.state_dict(),
            'accuracies': accuracies, 'train_accuracies': train_accuracies,
            'test_accuracies': test_accuracies,
            'weight_mag_log': weight_mag_log,
            'ranks': ranks, 'effective_ranks': effective_ranks,
            'approximate_ranks': approximate_ranks,
            'dead_neurons': dead_neurons, 'all_stable_rank': all_stable_rank,
            'all_effective_rank': all_effective_rank,
            'all_dormant_frac': all_dormant_frac,
            'task_metrics': task_metrics, 'params': config,
        }
        torch.save(ckpt_data, ckpt_file)
        elapsed = time.time() - wall_clock_start
        print(f"  Checkpoint saved at task {task_idx} ({reason}) [{elapsed/3600:.1f}h elapsed]")

    # Working MNIST copies (permuted in-place each task)
    x = x_mnist.clone()
    y = y_mnist.clone()
    x_test = x_test_mnist.clone()
    y_test = y_test_mnist.clone()
    time_limit_hit = False

    # ════════════════════════════════════════════════════════════════
    #  Task loop
    # ════════════════════════════════════════════════════════════════
    for task_idx in range(start_task, NUM_TASKS):
        t0 = time.time()
        elapsed = time.time() - wall_clock_start
        if elapsed > TIME_LIMIT_SECONDS:
            print(f"\n  Time limit ({elapsed/3600:.1f}h). Saving.")
            save_checkpoint(task_idx - 1, reason="time_limit")
            time_limit_hit = True; break

        iter_start = global_iter

        # ── 1. New pixel permutation + data shuffle ───────────────────
        rng        = np.random.RandomState(task_idx + SEED * 1000)
        pixel_perm = rng.permutation(INPUT_SIZE)
        data_perm  = rng.permutation(EXAMPLES_PER_TASK)
        x          = x[:, pixel_perm]
        x, y       = x[data_perm], y[data_perm]
        x_test     = x_test[:, pixel_perm]  # same permutation for test

        # ── 2. Reset K-FAC stats at task boundary ────────────────────
        if task_idx > 0:
            optimizer.hard_reset_stats()

        # ── 3. Pre-task probe: rank + dead neurons ────────────────────
        net.eval()
        with torch.no_grad():
            probe_x = x[:PROBE_SIZE].to(device)
            _, hidden_acts = net.predict(probe_x)
            for li in range(NUM_HIDDEN_LAYERS):
                act = hidden_acts[li]
                if not torch.isfinite(act).all():
                    ranks[task_idx][li] = float('nan')
                    effective_ranks[task_idx][li] = float('nan')
                    approximate_ranks[task_idx][li] = float('nan')
                    dead_neurons[task_idx][li] = 0.0
                    continue
                r, er, apr, _ = compute_matrix_rank_summaries(act, use_scipy=True)
                ranks[task_idx][li]             = r.float()
                effective_ranks[task_idx][li]   = er.float()
                approximate_ranks[task_idx][li] = apr.float()
                dead_neurons[task_idx][li] = (act.abs().sum(dim=0) == 0).sum().item()

        if task_idx % 10 == 0:
            print(f"  Task {task_idx:4d}  approx_rank={approximate_ranks[task_idx].tolist()}"
                  f"  dead={dead_neurons[task_idx].tolist()}")

        # ── 4. Train STEPS_PER_TASK mini-batch steps ──────────────────
        net.train()
        for step in range(STEPS_PER_TASK):
            start_idx = (step * MINI_BATCH_SIZE) % EXAMPLES_PER_TASK
            batch_x   = x[start_idx: start_idx + MINI_BATCH_SIZE].to(device)
            batch_y   = y[start_idx: start_idx + MINI_BATCH_SIZE].to(device)

            # K-FAC single-pass: forward → backward → natural gradient step
            optimizer.zero_grad()
            logits = net.predict(batch_x)[0]
            loss   = loss_fn(logits, batch_y)
            loss.backward()
            optimizer.step(loss_value=loss.item())
            optimizer.zero_grad(set_to_none=True)

            with torch.no_grad():
                out_sm = F.softmax(net.predict(batch_x)[0], dim=1)
                accuracies[global_iter] = accuracy(out_sm, batch_y).cpu()

                for l_idx, layer_idx in enumerate(net.layers_to_log):
                    if l_idx < weight_mag_log.shape[1]:
                        weight_mag_log[global_iter][l_idx] = (
                            net.layers[layer_idx].weight.data.abs().mean())

            global_iter += 1

        # ── 6. Train & Test evaluation ─────────────────────────────────
        net.eval()
        with torch.no_grad():
            # Train accuracy (full train set)
            train_correct = 0
            train_total = 0
            for si in range(0, x.shape[0], MINI_BATCH_SIZE):
                tb_x = x[si:si + MINI_BATCH_SIZE].to(device)
                tb_y = y[si:si + MINI_BATCH_SIZE].to(device)
                to = net.predict(tb_x)[0]
                train_correct += (to.argmax(dim=1) == tb_y).sum().item()
                train_total += tb_y.shape[0]
            train_acc = train_correct / max(train_total, 1)
            train_accuracies[task_idx] = train_acc

            # Test accuracy (EMA weights, full test set)
            test_correct = 0
            test_total = 0
            for si in range(0, x_test.shape[0], MINI_BATCH_SIZE):
                tb_x = x_test[si:si + MINI_BATCH_SIZE].to(device)
                tb_y = y_test[si:si + MINI_BATCH_SIZE].to(device)
                to = net.predict(tb_x)[0]
                test_correct += (to.argmax(dim=1) == tb_y).sum().item()
                test_total += tb_y.shape[0]
            test_acc = test_correct / max(test_total, 1)
            test_accuracies[task_idx] = test_acc

            # Dormant neurons + stable rank + effective rank (last hidden layer)
            probe_x = x[:PROBE_SIZE].to(device)
            agg_dormant, _, last_act = compute_dormant_neurons_ffnn(net, probe_x)
            all_dormant_frac[task_idx] = agg_dormant
            sr = compute_stable_rank_from_activations(last_act)
            all_stable_rank[task_idx] = sr
            er = compute_effective_rank_from_activations(last_act)
            all_effective_rank[task_idx] = er

        # ── 7. Per-task summary ────────────────────────────────────────
        task_acc = accuracies[iter_start:global_iter].mean().item()
        task_metrics['task_acc'].append(task_acc)
        task_metrics['task_train_acc'].append(train_acc)
        task_metrics['task_test_acc'].append(test_acc)
        task_metrics['task_id'].append(task_idx)
        task_metrics['avg_weight_mag'].append(
            weight_mag_log[iter_start:global_iter].mean(dim=0).tolist())

        et = time.time() - t0
        print(f"  [{method_name}] Task {task_idx:4d}/{NUM_TASKS}  "
              f"TrainAcc={train_acc:.4f}  TestAcc={test_acc:.4f}  "
              f"Dormant={agg_dormant:.3f}  SR={sr:.0f}  ER={er:.1f}  "
              f"AvgW={compute_avg_weight_magnitude(net):.4f}  "
              f"{et:.1f}s  λ={optimizer.damping:.2e}")

        if (task_idx + 1) % SAVE_EVERY_N_TASKS == 0 or task_idx == NUM_TASKS - 1:
            save_checkpoint(task_idx, reason="periodic" if task_idx < NUM_TASKS - 1 else "completed")

    # ── Save results ───────────────────────────────────────────────────
    result_file = os.path.join(RESULTS_DIR, f'{method_name}_results.pt')
    torch.save({
        'accuracies':        accuracies.cpu(),
        'train_accuracies':  train_accuracies.cpu(),
        'test_accuracies':   test_accuracies.cpu(),
        'weight_mag_log':    weight_mag_log.cpu(),
        'ranks':             ranks.cpu(),
        'effective_ranks':   effective_ranks.cpu(),
        'approximate_ranks': approximate_ranks.cpu(),
        'dead_neurons':      dead_neurons.cpu(),
        'all_stable_rank':   all_stable_rank.cpu(),
        'all_effective_rank': all_effective_rank.cpu(),
        'all_dormant_frac':  all_dormant_frac.cpu(),
        'task_metrics':      task_metrics,
    }, result_file)
    print(f"  ✓ Results saved to {result_file}")

    return (accuracies, train_accuracies, test_accuracies, weight_mag_log, ranks,
            effective_ranks, approximate_ranks, dead_neurons, all_stable_rank,
            all_effective_rank, all_dormant_frac, task_metrics)

print("✓ run_ngd training loop defined (Permuted MNIST)")

## 9b. CBP Training Loop (Nature 2024 baseline)
#
Continual Backpropagation: SGD + Generate-and-Test neuron replacement.

In [ ]:
def run_cbp(method_name, config):
    """CBP (Continual Backpropagation, Nature 2024) on Permuted MNIST."""
    print(f"\n{'='*70}")
    print(f"  {method_name} — Permuted MNIST ({NUM_TASKS} tasks)")
    print(f"  step_size={config['step_size']}, replacement_rate={config['replacement_rate']}")
    print(f"  opt={config['opt']}, decay_rate={config['decay_rate']}")
    print(f"{'='*70}")

    wall_clock_start = time.time()
    torch.manual_seed(SEED); np.random.seed(SEED)

    net = DeepFFNN(
        input_size=INPUT_SIZE, num_features=NUM_FEATURES,
        num_outputs=CLASSES_PER_TASK, num_hidden_layers=NUM_HIDDEN_LAYERS,
    ).to(device)

    learner = ContinualBackprop(
        net=net,
        step_size=config['step_size'],
        opt=config.get('opt', 'sgd'),
        loss='nll',
        replacement_rate=config['replacement_rate'],
        maturity_threshold=config.get('maturity_threshold', 100),
        decay_rate=config.get('decay_rate', 0.99),
        util_type=config.get('util_type', 'adaptable_contribution'),
        accumulate=config.get('accumulate', True),
        device=device,
    )

    # Metric tensors
    total_iters       = NUM_TASKS * STEPS_PER_TASK
    accuracies        = torch.zeros(total_iters, dtype=torch.float)
    train_accuracies  = torch.zeros(NUM_TASKS, dtype=torch.float)
    test_accuracies   = torch.zeros(NUM_TASKS, dtype=torch.float)
    weight_mag_log    = torch.zeros((total_iters, NUM_HIDDEN_LAYERS + 1), dtype=torch.float)
    ranks             = torch.zeros((NUM_TASKS, NUM_HIDDEN_LAYERS), dtype=torch.float)
    effective_ranks   = torch.zeros((NUM_TASKS, NUM_HIDDEN_LAYERS), dtype=torch.float)
    approximate_ranks = torch.zeros((NUM_TASKS, NUM_HIDDEN_LAYERS), dtype=torch.float)
    dead_neurons      = torch.zeros((NUM_TASKS, NUM_HIDDEN_LAYERS), dtype=torch.float)
    all_stable_rank   = torch.zeros(NUM_TASKS, dtype=torch.float)
    all_dormant_frac  = torch.zeros(NUM_TASKS, dtype=torch.float)
    task_metrics      = {'task_acc': [], 'task_train_acc': [], 'task_test_acc': [],
                         'task_id': [], 'avg_weight_mag': []}

    ckpt_file = _ckpt_path(method_name)
    start_task = 0
    global_iter = 0
    if os.path.isfile(ckpt_file):
        print(f"  Loading checkpoint: {ckpt_file}")
        ckpt = torch.load(ckpt_file, map_location=device, weights_only=False)
        net.load_state_dict(ckpt['model'])
        accuracies        = ckpt.get('accuracies',         accuracies)
        train_accuracies  = ckpt.get('train_accuracies',   train_accuracies)
        test_accuracies   = ckpt.get('test_accuracies',    test_accuracies)
        weight_mag_log    = ckpt.get('weight_mag_log',     weight_mag_log)
        ranks             = ckpt.get('ranks',              ranks)
        effective_ranks   = ckpt.get('effective_ranks',    effective_ranks)
        approximate_ranks = ckpt.get('approximate_ranks',  approximate_ranks)
        dead_neurons      = ckpt.get('dead_neurons',       dead_neurons)
        all_stable_rank   = ckpt.get('all_stable_rank',    all_stable_rank)
        all_dormant_frac  = ckpt.get('all_dormant_frac',   all_dormant_frac)
        task_metrics      = ckpt.get('task_metrics',       task_metrics)
        start_task        = ckpt['task'] + 1
        global_iter       = start_task * STEPS_PER_TASK
        # Rebuild GnT from restored net (GnT state not saved)
        learner = ContinualBackprop(
            net=net,
            step_size=config['step_size'],
            opt=config.get('opt', 'sgd'),
            loss='nll',
            replacement_rate=config['replacement_rate'],
            maturity_threshold=config.get('maturity_threshold', 100),
            decay_rate=config.get('decay_rate', 0.99),
            util_type=config.get('util_type', 'adaptable_contribution'),
            accumulate=config.get('accumulate', True),
            device=device,
        )
        print(f"  ✓ Resumed from task {start_task}")
        del ckpt; torch.cuda.empty_cache()
    else:
        print("  (no checkpoint — training from scratch)")

    def save_checkpoint_cbp(task_idx, reason="periodic"):
        ckpt_data = {
            'task': task_idx, 'model': net.state_dict(),
            'accuracies': accuracies, 'train_accuracies': train_accuracies,
            'test_accuracies': test_accuracies, 'weight_mag_log': weight_mag_log,
            'ranks': ranks, 'effective_ranks': effective_ranks,
            'approximate_ranks': approximate_ranks, 'dead_neurons': dead_neurons,
            'all_stable_rank': all_stable_rank, 'all_dormant_frac': all_dormant_frac,
            'task_metrics': task_metrics,
            'params': config,
        }
        torch.save(ckpt_data, ckpt_file)
        elapsed = time.time() - wall_clock_start
        print(f"  CBP checkpoint saved at task {task_idx} ({reason}) [{elapsed/3600:.1f}h elapsed]")

    x = x_mnist.clone()
    y = y_mnist.clone()
    x_test = x_test_mnist.clone()
    y_test = y_test_mnist.clone()

    for task_idx in range(start_task, NUM_TASKS):
        t0 = time.time()
        elapsed = time.time() - wall_clock_start
        if elapsed > TIME_LIMIT_SECONDS:
            print(f"\n  Time limit ({elapsed/3600:.1f}h). Saving.")
            save_checkpoint_cbp(task_idx - 1, reason="time_limit")
            break

        iter_start = global_iter

        # ── 1. Permutation ──
        rng        = np.random.RandomState(task_idx + SEED * 1000)
        pixel_perm = rng.permutation(INPUT_SIZE)
        data_perm  = rng.permutation(EXAMPLES_PER_TASK)
        x          = x[:, pixel_perm]
        x, y       = x[data_perm], y[data_perm]
        x_test     = x_test[:, pixel_perm]

        # ── 2. Pre-task probe: rank + dead neurons ──
        net.eval()
        with torch.no_grad():
            probe_x = x[:PROBE_SIZE].to(device)
            _, hidden_acts = net.predict(probe_x)
            for li in range(NUM_HIDDEN_LAYERS):
                act = hidden_acts[li]
                if not torch.isfinite(act).all():
                    ranks[task_idx][li] = float('nan')
                    effective_ranks[task_idx][li] = float('nan')
                    approximate_ranks[task_idx][li] = float('nan')
                    dead_neurons[task_idx][li] = 0.0
                    continue
                r, er, apr, _ = compute_matrix_rank_summaries(act, use_scipy=True)
                ranks[task_idx][li]             = r.float()
                effective_ranks[task_idx][li]   = er.float()
                approximate_ranks[task_idx][li] = apr.float()
                dead_neurons[task_idx][li] = (act.abs().sum(dim=0) == 0).sum().item()

        # ── 3. Train STEPS_PER_TASK steps ──
        net.train()
        for step in range(STEPS_PER_TASK):
            start_idx = (step * MINI_BATCH_SIZE) % EXAMPLES_PER_TASK
            batch_x   = x[start_idx: start_idx + MINI_BATCH_SIZE].to(device)
            batch_y   = y[start_idx: start_idx + MINI_BATCH_SIZE].to(device)

            loss, output = learner.learn(x=batch_x, target=batch_y)

            with torch.no_grad():
                out_sm = F.softmax(output, dim=1)
                accuracies[global_iter] = accuracy(out_sm, batch_y).cpu()
                for l_idx, layer_idx in enumerate(net.layers_to_log):
                    if l_idx < weight_mag_log.shape[1]:
                        weight_mag_log[global_iter][l_idx] = (
                            net.layers[layer_idx].weight.data.abs().mean())
            global_iter += 1

        # ── 4. Train & Test evaluation (raw weights, no EMA for CBP) ──
        net.eval()
        with torch.no_grad():
            train_correct, train_total = 0, 0
            for si in range(0, x.shape[0], MINI_BATCH_SIZE):
                tb_x = x[si:si + MINI_BATCH_SIZE].to(device)
                tb_y = y[si:si + MINI_BATCH_SIZE].to(device)
                to = net.predict(tb_x)[0]
                train_correct += (to.argmax(dim=1) == tb_y).sum().item()
                train_total += tb_y.shape[0]
            train_acc = train_correct / max(train_total, 1)
            train_accuracies[task_idx] = train_acc

            test_correct, test_total = 0, 0
            for si in range(0, x_test.shape[0], MINI_BATCH_SIZE):
                tb_x = x_test[si:si + MINI_BATCH_SIZE].to(device)
                tb_y = y_test[si:si + MINI_BATCH_SIZE].to(device)
                to = net.predict(tb_x)[0]
                test_correct += (to.argmax(dim=1) == tb_y).sum().item()
                test_total += tb_y.shape[0]
            test_acc = test_correct / max(test_total, 1)
            test_accuracies[task_idx] = test_acc

            # Dormant neurons + stable rank
            probe_x = x[:PROBE_SIZE].to(device)
            agg_dormant, _, last_act = compute_dormant_neurons_ffnn(net, probe_x)
            all_dormant_frac[task_idx] = agg_dormant
            sr = compute_stable_rank_from_activations(last_act)
            all_stable_rank[task_idx] = sr

        # ── 5. Per-task summary ──
        task_acc = accuracies[iter_start:global_iter].mean().item()
        task_metrics['task_acc'].append(task_acc)
        task_metrics['task_train_acc'].append(train_acc)
        task_metrics['task_test_acc'].append(test_acc)
        task_metrics['task_id'].append(task_idx)
        task_metrics['avg_weight_mag'].append(
            weight_mag_log[iter_start:global_iter].mean(dim=0).tolist())

        et = time.time() - t0
        print(f"  [{method_name}] Task {task_idx:4d}/{NUM_TASKS}  "
              f"TrainAcc={train_acc:.4f}  TestAcc={test_acc:.4f}  "
              f"Dormant={agg_dormant:.3f}  SR={sr:.0f}  "
              f"AvgW={compute_avg_weight_magnitude(net):.4f}  "
              f"{et:.1f}s")

        if (task_idx + 1) % SAVE_EVERY_N_TASKS == 0 or task_idx == NUM_TASKS - 1:
            save_checkpoint_cbp(task_idx, reason="periodic" if task_idx < NUM_TASKS - 1 else "completed")

    result_file = os.path.join(RESULTS_DIR, f'{method_name}_results.pt')
    torch.save({
        'accuracies':        accuracies.cpu(),
        'train_accuracies':  train_accuracies.cpu(),
        'test_accuracies':   test_accuracies.cpu(),
        'weight_mag_log':    weight_mag_log.cpu(),
        'ranks':             ranks.cpu(),
        'effective_ranks':   effective_ranks.cpu(),
        'approximate_ranks': approximate_ranks.cpu(),
        'dead_neurons':      dead_neurons.cpu(),
        'all_stable_rank':   all_stable_rank.cpu(),
        'all_dormant_frac':  all_dormant_frac.cpu(),
        'task_metrics':      task_metrics,
    }, result_file)
    print(f"  ✓ CBP results saved to {result_file}")

    return (accuracies, train_accuracies, test_accuracies, weight_mag_log, ranks,
            effective_ranks, approximate_ranks, dead_neurons, all_stable_rank,
            torch.zeros(NUM_TASKS, dtype=torch.float),  # all_effective_rank (N/A for CBP)
            all_dormant_frac, task_metrics)

print("✓ run_cbp training loop defined (Permuted MNIST)")

## 10. Run Experiments

In [ ]:
all_results = {}
for method in METHODS_TO_RUN:
    cfg = CONFIGS[method]
    if method == 'cbp':
        result = run_cbp(method, cfg)
    else:
        result = run_ngd(method, cfg)
    # Unpack: (acc, train_acc, test_acc, wmag, ranks, er_layer, apr, dead, sr, eff_rank, dormant, task)
    all_results[method] = {
        'acc': result[0], 'train_acc': result[1], 'test_acc': result[2],
        'wmag': result[3], 'ranks': result[4], 'er': result[5],
        'apr': result[6], 'dead': result[7], 'sr': result[8],
        'eff_rank': result[9], 'dormant': result[10], 'task': result[11],
    }

## 11. Results Plots

In [ ]:
TASK_WINDOW = 50
LAYER_NAMES = [f'Hidden {i+1} ({NUM_FEATURES})' for i in range(NUM_HIDDEN_LAYERS)]

METHOD_STYLES = {
    'ngd_kfac': {'color': '#2196F3', 'label': 'NGD (K-FAC)'},
    'cbp':      {'color': '#FF5722', 'label': 'CBP (Nature 2024)'},
}

def smooth(arr, w=TASK_WINDOW):
    n = len(arr) // w
    return np.array([arr[i*w:(i+1)*w].mean() for i in range(n)])

def _style(method):
    return METHOD_STYLES.get(method, {'color': 'gray', 'label': method})

# ── Comparison Figure: 4×3 grid ──
fig, axes = plt.subplots(4, 3, figsize=(20, 22))
fig.suptitle('NGD (K-FAC) vs CBP — Continual Permuted MNIST\n'
             f'Network: {INPUT_SIZE}→{NUM_FEATURES}×{NUM_HIDDEN_LAYERS}→{CLASSES_PER_TASK}, '
             f'{NUM_TASKS} tasks, batch={MINI_BATCH_SIZE}',
             fontsize=14, fontweight='bold')

def _clean(ax):
    ax.grid(True, alpha=0.25); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# ── Row 0, Col 0: Test Accuracy ──
ax = axes[0, 0]
for m, d in all_results.items():
    s = _style(m)
    test_arr = np.array(d['task']['task_test_acc'])
    ax.plot(test_arr * 100, color=s['color'], lw=0.8, alpha=0.3)
    sm = smooth(test_arr) * 100
    ax.plot(np.arange(len(sm)) * TASK_WINDOW, sm, color=s['color'], lw=2.5, label=s['label'])
ax.set_xlabel('Task'); ax.set_ylabel('Accuracy (%)'); ax.set_title('Test Accuracy')
ax.legend(fontsize=9); _clean(ax)

# ── Row 0, Col 1: Train Accuracy ──
ax = axes[0, 1]
for m, d in all_results.items():
    s = _style(m)
    train_arr = np.array(d['task']['task_train_acc'])
    ax.plot(train_arr * 100, color=s['color'], lw=0.8, alpha=0.3)
    sm = smooth(train_arr) * 100
    ax.plot(np.arange(len(sm)) * TASK_WINDOW, sm, color=s['color'], lw=2.5, label=s['label'])
ax.set_xlabel('Task'); ax.set_ylabel('Accuracy (%)'); ax.set_title('Train Accuracy')
ax.legend(fontsize=9); _clean(ax)

# ── Row 0, Col 2: Train-Test Gap ──
ax = axes[0, 2]
for m, d in all_results.items():
    s = _style(m)
    tr = np.array(d['task']['task_train_acc'])
    te = np.array(d['task']['task_test_acc'])
    gap = (tr - te) * 100
    sm = smooth(gap)
    ax.plot(np.arange(len(sm)) * TASK_WINDOW, sm, color=s['color'], lw=2.5, label=s['label'])
ax.set_xlabel('Task'); ax.set_ylabel('Gap (%)'); ax.set_title('Train − Test Gap')
ax.axhline(0, color='black', ls=':', lw=0.8, alpha=0.5)
ax.legend(fontsize=9); _clean(ax)

# ── Row 1, Col 0: Stable Rank ──
ax = axes[1, 0]
for m, d in all_results.items():
    s = _style(m)
    sr = d['sr'].cpu().numpy()
    ax.plot(sr, color=s['color'], lw=0.8, alpha=0.3)
    sm = smooth(sr)
    ax.plot(np.arange(len(sm)) * TASK_WINDOW, sm, color=s['color'], lw=2.5, label=s['label'])
ax.set_xlabel('Task'); ax.set_ylabel('Stable Rank'); ax.set_title('Stable Rank (last hidden layer)')
ax.legend(fontsize=9); _clean(ax)

# ── Row 1, Col 1: Dormant Neuron Fraction ──
ax = axes[1, 1]
for m, d in all_results.items():
    s = _style(m)
    dorm = d['dormant'].cpu().numpy() * 100
    sm = smooth(dorm)
    ax.plot(np.arange(len(sm)) * TASK_WINDOW, sm, color=s['color'], lw=2.5, label=s['label'])
ax.set_xlabel('Task'); ax.set_ylabel('Dormant (%)'); ax.set_title('Dormant Neuron Fraction')
ax.legend(fontsize=9); _clean(ax)

# ── Row 1, Col 2: Weight Magnitude (Layer 1) ──
ax = axes[1, 2]
for m, d in all_results.items():
    s = _style(m)
    wmag = [d['task']['avg_weight_mag'][t][0] if t < len(d['task']['avg_weight_mag']) else 0
            for t in range(len(d['task']['task_acc']))]
    ax.plot(wmag, color=s['color'], lw=1.5, label=s['label'])
ax.set_xlabel('Task'); ax.set_ylabel('Avg |W|'); ax.set_title('Weight Magnitude (Layer 1)')
ax.legend(fontsize=9); _clean(ax)

# ── Row 2, Col 0: Avg Approximate Rank ──
ax = axes[2, 0]
for m, d in all_results.items():
    s = _style(m)
    apr_avg = d['apr'].mean(dim=1).numpy()
    sm = smooth(apr_avg)
    ax.plot(np.arange(len(sm)) * TASK_WINDOW, sm, color=s['color'], lw=2.5, label=s['label'])
ax.set_xlabel('Task'); ax.set_ylabel('Approx Rank'); ax.set_title('Avg Approximate Rank (all layers)')
ax.legend(fontsize=9); _clean(ax)

# ── Row 2, Col 1: Total Dead Neurons ──
ax = axes[2, 1]
for m, d in all_results.items():
    s = _style(m)
    dead_total = d['dead'].sum(dim=1).numpy()
    sm = smooth(dead_total)
    ax.plot(np.arange(len(sm)) * TASK_WINDOW, sm, color=s['color'], lw=2.5, label=s['label'])
ax.set_xlabel('Task'); ax.set_ylabel('Dead Neurons'); ax.set_title('Total Dead Neurons (all layers)')
ax.legend(fontsize=9); _clean(ax)

# ── Row 2, Col 2: Summary Table ──
ax = axes[2, 2]
ax.axis('off')
n_final = 100
_m1, _m2 = 'ngd_kfac', 'cbp'
_l1 = _style(_m1)['label']
lines = ['Final 100-task metrics:', '─' * 46,
         f'{"":<18} {_l1:>14} {"CBP":>10}']
for metric_name, metric_fn in [
    ('Test Acc (%)',   lambda d: np.array(d['task']['task_test_acc'][-n_final:]).mean() * 100),
    ('Train Acc (%)',  lambda d: np.array(d['task']['task_train_acc'][-n_final:]).mean() * 100),
    ('Stable Rank',    lambda d: d['sr'][-n_final:].mean().item()),
    ('Eff. Rank',      lambda d: d['eff_rank'][-n_final:].mean().item()),
    ('Dormant (%)',    lambda d: d['dormant'][-n_final:].mean().item() * 100),
    ('Dead Neurons',   lambda d: d['dead'][-n_final:].sum(dim=1).mean().item()),
    ('Approx Rank',    lambda d: d['apr'][-n_final:].mean().item()),
]:
    vals = []
    for m in [_m1, _m2]:
        if m in all_results:
            vals.append(f'{metric_fn(all_results[m]):>10.2f}')
        else:
            vals.append(f'{"N/A":>10}')
    lines.append(f'{metric_name:<18} {vals[0]:>14} {vals[1]:>10}')
ax.text(0.5, 0.5, '\n'.join(lines), transform=ax.transAxes,
        fontsize=10, verticalalignment='center', horizontalalignment='center',
        family='monospace',
        bbox=dict(boxstyle='round,pad=0.8', facecolor='#FFF9C4', edgecolor='#FBC02D', alpha=0.85))

# ── Row 3, Col 0: Effective Rank (last hidden layer) ──
ax = axes[3, 0]
for m, d in all_results.items():
    s = _style(m)
    er = d['eff_rank'].cpu().numpy()
    ax.plot(er, color=s['color'], lw=0.8, alpha=0.3)
    sm = smooth(er)
    ax.plot(np.arange(len(sm)) * TASK_WINDOW, sm, color=s['color'], lw=2.5, label=s['label'])
ax.set_xlabel('Task'); ax.set_ylabel('Effective Rank'); ax.set_title('Effective Rank (last hidden layer)')
ax.legend(fontsize=9); _clean(ax)

# ── Row 3, Col 1: Avg Effective Rank (all layers) ──
ax = axes[3, 1]
for m, d in all_results.items():
    s = _style(m)
    er_avg = d['er'].mean(dim=1).numpy()
    sm = smooth(er_avg)
    ax.plot(np.arange(len(sm)) * TASK_WINDOW, sm, color=s['color'], lw=2.5, label=s['label'])
ax.set_xlabel('Task'); ax.set_ylabel('Effective Rank'); ax.set_title('Avg Effective Rank (all layers)')
ax.legend(fontsize=9); _clean(ax)

# ── Row 3, Col 2: unused ──
axes[3, 2].axis('off')

plt.tight_layout()
plot_file = os.path.join(RESULTS_DIR, 'ngd_vs_cbp_comparison.png')
plt.savefig(plot_file, dpi=200, bbox_inches='tight')
plt.show()
print(f"✓ Comparison plot saved to {plot_file}")

## 12. Per-Layer Detail: Approximate Rank

In [ ]:
fig2, axes2 = plt.subplots(2, NUM_HIDDEN_LAYERS, figsize=(5 * NUM_HIDDEN_LAYERS, 8), sharey='row')
fig2.suptitle('Approximate Rank per Hidden Layer — NGD (K-FAC) vs CBP',
              fontsize=13, fontweight='bold')
for row_idx, (m, label) in enumerate([('ngd_kfac', 'NGD (K-FAC)'), ('cbp', 'CBP')]):
    if m not in all_results: continue
    d = all_results[m]
    for li in range(NUM_HIDDEN_LAYERS):
        ax = axes2[row_idx, li]
        ax.plot(d['apr'][:, li].numpy(), color=_style(m)['color'], lw=1.5)
        ax.set_title(f'{label} — {LAYER_NAMES[li]}', fontsize=10)
        ax.set_xlabel('Task')
        if li == 0: ax.set_ylabel('Approx Rank')
        _clean(ax)
plt.tight_layout()
plot_file2 = os.path.join(RESULTS_DIR, 'rank_evolution_comparison.png')
plt.savefig(plot_file2, dpi=200, bbox_inches='tight'); plt.show()
print(f"✓ Rank evolution saved to {plot_file2}")

## 13. Per-Layer Detail: Dead Neurons

In [ ]:
fig3, axes3 = plt.subplots(2, NUM_HIDDEN_LAYERS, figsize=(5 * NUM_HIDDEN_LAYERS, 8), sharey='row')
fig3.suptitle('Dead Neurons per Hidden Layer — NGD (K-FAC) vs CBP',
              fontsize=13, fontweight='bold')
for row_idx, (m, label) in enumerate([('ngd_kfac', 'NGD (K-FAC)'), ('cbp', 'CBP')]):
    if m not in all_results: continue
    d = all_results[m]
    for li in range(NUM_HIDDEN_LAYERS):
        ax = axes3[row_idx, li]
        ax.plot(d['dead'][:, li].numpy(), color=_style(m)['color'], lw=1.5)
        ax.set_title(f'{label} — {LAYER_NAMES[li]}', fontsize=10)
        ax.set_xlabel('Task')
        if li == 0: ax.set_ylabel('Dead Neurons')
        _clean(ax)
plt.tight_layout()
plot_file3 = os.path.join(RESULTS_DIR, 'dead_neurons_comparison.png')
plt.savefig(plot_file3, dpi=200, bbox_inches='tight'); plt.show()
print(f"✓ Dead neurons saved to {plot_file3}")

## 14. Summary

In [ ]:
_ngd_label = _style('ngd_kfac')['label']
print(f"\n{'='*75}")
print(f"  Comparison — Permuted MNIST — Final {n_final}-task avg:")
print(f"{'='*75}")
print(f"{'Metric':<20} {_ngd_label:>20} {'CBP':>10}")
print(f"{'─'*52}")
for label, fn in [
    ('Test Acc (%)',   lambda d: np.array(d['task']['task_test_acc'][-n_final:]).mean() * 100),
    ('Train Acc (%)',  lambda d: np.array(d['task']['task_train_acc'][-n_final:]).mean() * 100),
    ('Stable Rank',    lambda d: d['sr'][-n_final:].mean().item()),
    ('Eff. Rank',      lambda d: d['eff_rank'][-n_final:].mean().item()),
    ('Dormant (%)',    lambda d: d['dormant'][-n_final:].mean().item() * 100),
    ('Dead Neurons',   lambda d: d['dead'][-n_final:].sum(dim=1).mean().item()),
    ('Approx Rank',    lambda d: d['apr'][-n_final:].mean().item()),
    ('Weight Mag',     lambda d: np.mean([d['task']['avg_weight_mag'][t][0]
                              for t in range(-n_final, 0) if t + len(d['task']['avg_weight_mag']) >= 0])),
]:
    vals = []
    for m in ['ngd_kfac', 'cbp']:
        if m in all_results:
            vals.append(f'{fn(all_results[m]):>10.2f}')
        else:
            vals.append(f'{"N/A":>10}')
    print(f"{label:<20} {vals[0]:>20} {vals[1]:>10}")
print(f"{'='*75}")